# Notebook 4 — Comparación final de los tres modelos

**Proyecto:** Sistema de pronóstico de demanda eléctrica en Ecuador

Este notebook consolida los resultados de los tres modelos (SARIMA, Prophet, LSTM) y genera la tabla comparativa final, los gráficos de comparación y el resumen ejecutivo para la sección de Resultados del artículo.

**Requisito:** ejecutar primero los Notebooks 0, 1, 2 y 3.


## 1. Importaciones

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.family"] = "serif"


## 2. Carga de resultados de los tres modelos

In [ ]:
sarima  = pd.read_csv("resultados_sarima.csv",  parse_dates=["fecha"])
prophet = pd.read_csv("resultados_prophet.csv", parse_dates=["fecha"])
lstm    = pd.read_csv("resultados_lstm.csv",    parse_dates=["fecha"])

metr_sarima  = pd.read_csv("metricas_sarima.csv")
metr_prophet = pd.read_csv("metricas_prophet.csv")
metr_lstm    = pd.read_csv("metricas_lstm.csv")

with open("config_sarima.json") as f:  cfg_sarima  = json.load(f)
with open("config_prophet.json") as f: cfg_prophet = json.load(f)
with open("config_lstm.json") as f:    cfg_lstm    = json.load(f)

print("Configuraciones finales de cada modelo:\n")
print(f"  SARIMA:  order={cfg_sarima['order']}, seasonal_order={cfg_sarima['seasonal_order']}")
print(f"  Prophet: mode={cfg_prophet['seasonality_mode']}, "
      f"cp_prior={cfg_prophet['changepoint_prior_scale']}, "
      f"seas_prior={cfg_prophet['seasonality_prior_scale']}")
print(f"  LSTM:    unidades={cfg_lstm['unidades']}, dropout={cfg_lstm['dropout']}, "
      f"lr={cfg_lstm['learning_rate']}, batch={cfg_lstm['batch_size']}")


## 3. Tabla comparativa de métricas sobre el conjunto de prueba

In [ ]:
# SARIMA - test
ms = metr_sarima[metr_sarima["conjunto"]=="test"].iloc[0]
# Prophet - test
mp = metr_prophet[metr_prophet["conjunto"]=="test"].iloc[0]
# LSTM - estructura distinta (pivot)
ml = metr_lstm.set_index("métrica")

tabla = pd.DataFrame({
    "MAE (GWh)":  [ms["MAE"],  mp["MAE"],  ml.loc["MAE","pred_promedio"]],
    "RMSE (GWh)": [ms["RMSE"], mp["RMSE"], ml.loc["RMSE","pred_promedio"]],
    "MAPE (%)":   [ms["MAPE"], mp["MAPE"], ml.loc["MAPE","pred_promedio"]],
}, index=["SARIMA", "Prophet", "LSTM"]).round(2)

print("Comparación sobre el conjunto de prueba (abr-dic 2024):\n")
print(tabla.to_string())

# Identificar el ganador por cada métrica
print("\nMejor modelo por métrica:")
for m in tabla.columns:
    ganador = tabla[m].idxmin()
    print(f"  {m:12s} -> {ganador}  ({tabla.loc[ganador, m]:.2f})")


## 4. Gráfico de barras comparativo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colores = {"SARIMA": "#1f4e79", "Prophet": "#2e7d32", "LSTM": "#c62828"}
metricas = ["MAE (GWh)", "RMSE (GWh)", "MAPE (%)"]

for ax, m in zip(axes, metricas):
    valores = tabla[m]
    ganador = valores.idxmin()
    barras_color = [colores[mod] if mod != ganador else "#ffa000" for mod in tabla.index]
    bars = ax.bar(tabla.index, valores, color=barras_color, edgecolor="black", lw=0.6)
    for b, v in zip(bars, valores):
        ax.text(b.get_x() + b.get_width()/2, b.get_height(),
                f"{v:.2f}", ha="center", va="bottom", fontsize=10)
    ax.set_title(m)
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Comparación de modelos sobre el conjunto de prueba  (mejor en naranja)",
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig("fig_comparacion_metricas.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Gráfico de predicciones superpuestas

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))

# Real
ax.plot(sarima["fecha"], sarima["real_gwh"], label="Real",
        color="black", lw=2.0, marker="o", ms=6)

# Modelos
ax.plot(sarima["fecha"],  sarima["sarima_pred_gwh"],  label="SARIMA",
        color="#1f4e79", lw=1.8, marker="s", ms=5, linestyle="--")
ax.plot(prophet["fecha"], prophet["prophet_pred_gwh"], label="Prophet",
        color="#2e7d32", lw=1.8, marker="^", ms=5, linestyle="--")
ax.plot(lstm["fecha"],    lstm["lstm_pred_media_gwh"], label="LSTM (media)",
        color="#c62828", lw=1.8, marker="d", ms=5, linestyle="--")

ax.fill_between(lstm["fecha"],
                lstm["lstm_pred_media_gwh"] - lstm["lstm_pred_std_gwh"],
                lstm["lstm_pred_media_gwh"] + lstm["lstm_pred_std_gwh"],
                color="#c62828", alpha=0.12, label="LSTM ± 1 std")

ax.set_title("Predicciones de los tres modelos sobre el conjunto de prueba (abr-dic 2024)")
ax.set_ylabel("Demanda (GWh)"); ax.set_xlabel("Fecha"); ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_comparacion_predicciones.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Errores absolutos mes a mes

In [ ]:
err = pd.DataFrame({
    "fecha": sarima["fecha"],
    "Real": sarima["real_gwh"],
    "Err_SARIMA":  np.abs(sarima["real_gwh"]  - sarima["sarima_pred_gwh"]),
    "Err_Prophet": np.abs(prophet["real_gwh"] - prophet["prophet_pred_gwh"]),
    "Err_LSTM":    np.abs(lstm["real_gwh"]    - lstm["lstm_pred_media_gwh"]),
})

fig, ax = plt.subplots(figsize=(13, 4.5))
x = np.arange(len(err))
w = 0.27
ax.bar(x - w, err["Err_SARIMA"],  width=w, label="SARIMA",  color="#1f4e79")
ax.bar(x,     err["Err_Prophet"], width=w, label="Prophet", color="#2e7d32")
ax.bar(x + w, err["Err_LSTM"],    width=w, label="LSTM",    color="#c62828")
ax.set_xticks(x)
ax.set_xticklabels([d.strftime("%b-%y") for d in err["fecha"]], rotation=45)
ax.set_ylabel("Error absoluto (GWh)")
ax.set_title("Error absoluto mensual por modelo")
ax.legend(); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig_errores_mensuales.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nResumen de errores absolutos (GWh):")
print(err[["Err_SARIMA","Err_Prophet","Err_LSTM"]].describe().round(2).to_string())


## 7. Tabla completa para el artículo

In [ ]:
# Construir la tabla final con todos los detalles
detalle = pd.DataFrame({
    "Fecha":   sarima["fecha"].dt.strftime("%Y-%m"),
    "Real":    sarima["real_gwh"].round(2),
    "SARIMA":  sarima["sarima_pred_gwh"].round(2),
    "Prophet": prophet["prophet_pred_gwh"].round(2),
    "LSTM":    lstm["lstm_pred_media_gwh"].round(2),
})
print("Predicciones mes a mes (todos los modelos):\n")
print(detalle.to_string(index=False))

detalle.to_csv("tabla_predicciones_completa.csv", index=False)
tabla.to_csv("tabla_metricas_comparativa.csv")


## 8. Resumen ejecutivo

In [ ]:
ganador_por_metrica = {m: tabla[m].idxmin() for m in tabla.columns}
ganador_global = pd.Series(list(ganador_por_metrica.values())).mode().iloc[0]

print("=" * 60)
print("RESUMEN EJECUTIVO")
print("=" * 60)
print(f"\nPeríodo: 2020-01 a 2024-12 (60 observaciones mensuales)")
print(f"Test:    abril-diciembre 2024 (9 observaciones)\n")
print("Configuración final de cada modelo:")
print(f"  SARIMA{tuple(cfg_sarima['order'])} x {tuple(cfg_sarima['seasonal_order'])}")
print(f"  Prophet ({cfg_prophet['seasonality_mode']}, cp_prior={cfg_prophet['changepoint_prior_scale']})")
print(f"  LSTM ({cfg_lstm['unidades']} unidades, dropout={cfg_lstm['dropout']}, "
      f"prom. {cfg_lstm['n_corridas']} corridas)")
print(f"\nMétricas sobre test:")
print(tabla.to_string())
print(f"\nModelo ganador por métrica:")
for m, mod in ganador_por_metrica.items():
    print(f"  {m:12s} -> {mod}")
print(f"\nModelo más consistente (gana en más métricas): {ganador_global}")
print("\n" + "=" * 60)

# Guardar el resumen en JSON para referencia
import json
resumen = {
    "periodo_train": f"{cfg_sarima.get('order','')}",
    "ganador_por_metrica": ganador_por_metrica,
    "ganador_global": ganador_global,
    "metricas_test": tabla.to_dict()
}
with open("resumen_comparacion.json", "w") as f:
    json.dump(resumen, f, indent=2)

print("\nArchivos finales generados:")
print("  tabla_metricas_comparativa.csv")
print("  tabla_predicciones_completa.csv")
print("  resumen_comparacion.json")
print("  fig_comparacion_metricas.png")
print("  fig_comparacion_predicciones.png")
print("  fig_errores_mensuales.png")
